In [1]:
"""
数据质量检核引擎 - 生产风格版本
功能：对CSV文件执行质量检核，输出清洗后数据和HTML报告
"""

import pandas as pd
import numpy as np
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

class DataQualityEngine:
    """数据质量检核引擎"""
    
    def __init__(self, config=None):
        """
        初始化引擎
        config: 规则配置文件（dict），如果为None则使用默认规则
        """
        self.config = config or self._get_default_config()
        self.issues_log = []
        self.original_df = None
        self.cleaned_df = None
        
    def _get_default_config(self):
        """默认的检核规则配置"""
        return {
            'rules': [
                {
                    'name': 'VendorID非空',
                    'column': 'VendorID',
                    'rule_type': 'not_null',
                    'severity': 'high',
                    'action': 'fill_with_mode'
                },
                {
                    'name': 'pickup_datetime非空',
                    'column': 'pickup_datetime',
                    'rule_type': 'not_null',
                    'severity': 'high',
                    'action': 'drop_row'
                },
                {
                    'name': 'trip_distance大于0',
                    'column': 'trip_distance',
                    'rule_type': 'greater_than',
                    'threshold': 0,
                    'severity': 'medium',
                    'action': 'fill_with_median'
                },
                {
                    'name': 'fare_amount非负',
                    'column': 'fare_amount',
                    'rule_type': 'greater_equal',
                    'threshold': 0,
                    'severity': 'high',
                    'action': 'abs_value'
                },
                {
                    'name': 'passenger_count范围[1,6]',
                    'column': 'passenger_count',
                    'rule_type': 'range',
                    'min': 1,
                    'max': 6,
                    'severity': 'medium',
                    'action': 'clip'
                },
                {
                    'name': 'payment_type枚举值',
                    'column': 'payment_type',
                    'rule_type': 'in_list',
                    'valid_values': [1, 2, 3, 4, 5],
                    'severity': 'medium',
                    'action': 'fill_with_mode'
                }
            ],
            'remove_duplicates': True
        }
    
    def _check_rule(self, df, rule):
        """执行单条检核规则，返回违反规则的行索引"""
        rule_type = rule['rule_type']
        
        if rule_type == 'not_null':
            return df[rule['column']].isnull()
        
        elif rule_type == 'greater_than':
            return df[rule['column']] <= rule['threshold']
        
        elif rule_type == 'greater_equal':
            return df[rule['column']] < rule['threshold']
        
        elif rule_type == 'range':
            return (df[rule['column']] < rule['min']) | (df[rule['column']] > rule['max'])
        
        elif rule_type == 'in_list':
            return ~df[rule['column']].isin(rule['valid_values'])
        
        return pd.Series([False] * len(df))
    
    def _apply_fix(self, df, rule, violated_mask):
        """根据规则配置执行修复"""
        action = rule['action']
        column = rule['column']
        
        if action == 'fill_with_mode':
            mode_val = df[column].mode()
            fill_val = mode_val.iloc[0] if len(mode_val) > 0 else 0
            df.loc[violated_mask, column] = fill_val
            return fill_val
        
        elif action == 'fill_with_median':
            median_val = df[~violated_mask][column].median()
            df.loc[violated_mask, column] = median_val
            return median_val
        
        elif action == 'drop_row':
            return None  # 行删除在外层处理
        
        elif action == 'abs_value':
            df.loc[violated_mask, column] = df.loc[violated_mask, column].abs()
            return 'abs'
        
        elif action == 'clip':
            min_val = rule.get('min', -float('inf'))
            max_val = rule.get('max', float('inf'))
            df.loc[violated_mask, column] = df.loc[violated_mask, column].clip(min_val, max_val)
            return f'clip to [{min_val}, {max_val}]'
        
        return None
    
    def run_checks(self, df):
        """执行所有检核规则"""
        self.original_df = df.copy()
        self.cleaned_df = df.copy()
        self.issues_log = []
        
        print("🔍 执行质量检核...")
        for rule in self.config['rules']:
            violated = self._check_rule(self.cleaned_df, rule)
            count = violated.sum()
            
            self.issues_log.append({
                'rule_name': rule['name'],
                'column': rule['column'],
                'severity': rule['severity'],
                'violated_count': count,
                'violated_rows': violated[violated].index.tolist()[:10]  # 只记录前10个
            })
            print(f"   {rule['name']}: {count} 行违反")
        
        return self.issues_log
    
    def apply_fixes(self):
        """执行所有修复操作"""
        print("\n🔧 执行数据修复...")
        
        # 记录修复操作日志
        fix_log = []
        
        # 先处理需要删除行的规则
        rows_to_drop = pd.Series([False] * len(self.cleaned_df))
        
        for rule in self.config['rules']:
            if rule['action'] == 'drop_row':
                violated = self._check_rule(self.cleaned_df, rule)
                rows_to_drop = rows_to_drop | violated
        
        if rows_to_drop.any():
            print(f"   删除行: {rows_to_drop.sum()} 行")
            self.cleaned_df = self.cleaned_df[~rows_to_drop]
            fix_log.append(f"删除 {rows_to_drop.sum()} 行")
        
        # 重新索引
        self.cleaned_df = self.cleaned_df.reset_index(drop=True)
        
        # 处理其他修复
        for rule in self.config['rules']:
            if rule['action'] != 'drop_row':
                violated = self._check_rule(self.cleaned_df, rule)
                if violated.any():
                    fix_result = self._apply_fix(self.cleaned_df, rule, violated)
                    fix_log.append(f"{rule['name']}: 修复 {violated.sum()} 行 -> {fix_result}")
                    print(f"   {rule['name']}: 修复 {violated.sum()} 行")
        
        # 去重
        if self.config.get('remove_duplicates', True):
            before = len(self.cleaned_df)
            self.cleaned_df = self.cleaned_df.drop_duplicates()
            after = len(self.cleaned_df)
            if before - after > 0:
                print(f"   去重: 删除 {before - after} 行")
                fix_log.append(f"删除重复行: {before - after} 行")
        
        return fix_log
    
    def generate_report(self, output_path='quality_report.html'):
        """生成HTML质量报告"""
        original_count = len(self.original_df)
        cleaned_count = len(self.cleaned_df)
        
        html = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <meta charset="UTF-8">
            <title>数据质量报告</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                h1 {{ color: #2c3e50; }}
                .summary {{ background: #ecf0f1; padding: 15px; border-radius: 8px; margin-bottom: 20px; }}
                table {{ border-collapse: collapse; width: 100%; margin-top: 10px; }}
                th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
                th {{ background: #3498db; color: white; }}
                .high {{ color: #e74c3c; font-weight: bold; }}
                .medium {{ color: #f39c12; }}
                .low {{ color: #27ae60; }}
                .footer {{ margin-top: 30px; font-size: 12px; color: #7f8c8d; }}
            </style>
        </head>
        <body>
            <h1>📊 数据质量报告</h1>
            <div class="summary">
                <h2>执行摘要</h2>
                <p><strong>执行时间:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                <p><strong>原始数据行数:</strong> {original_count}</p>
                <p><strong>清洗后行数:</strong> {cleaned_count}</p>
                <p><strong>处理行数:</strong> {original_count - cleaned_count}</p>
                <p><strong>检核规则数:</strong> {len(self.issues_log)}</p>
            </div>
            
            <h2>质量检核结果</h2>
            <table>
                <tr>
                    <th>规则名称</th>
                    <th>字段</th>
                    <th>严重级别</th>
                    <th>违反行数</th>
                </tr>
        """
        
        for issue in self.issues_log:
            severity_class = issue['severity']
            html += f"""
                <tr>
                    <td>{issue['rule_name']}</td>
                    <td>{issue['column']}</td>
                    <td class="{severity_class}">{issue['severity'].upper()}</td>
                    <td>{issue['violated_count']}</td>
                </tr>
            """
        
        html += f"""
            </table>
            
            <h2>清洗后数据样例</h2>
            {self.cleaned_df.head(10).to_html()}
            
            <div class="footer">
                <p>报告由 DataQualityEngine 自动生成 | 清洗后数据保存至 nyc_taxi_clean.csv</p>
            </div>
        </body>
        </html>
        """
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html)
        print(f"\n📄 质量报告已生成: {output_path}")
        
        return html
    
    def run(self, input_path, output_path='nyc_taxi_clean.csv'):
        """完整的运行流程"""
        print("=" * 50)
        print("数据质量检核引擎 v1.0")
        print("=" * 50)
        
        # 加载数据
        print(f"\n📂 加载数据: {input_path}")
        df = pd.read_csv(input_path, parse_dates=['pickup_datetime', 'dropoff_datetime'])
        print(f"   数据形状: {df.shape}")
        
        # 执行检核
        self.run_checks(df)
        
        # 执行修复
        fix_log = self.apply_fixes()
        
        # 保存清洗后数据
        self.cleaned_df.to_csv(output_path, index=False)
        print(f"\n💾 清洗后数据保存至: {output_path}")
        
        # 生成报告
        self.generate_report()
        
        print("\n✅ 数据质量治理完成！")
        return self.cleaned_df


if __name__ == "__main__":
    # 使用示例
    engine = DataQualityEngine()
    result = engine.run('nyc_taxi_dirty.csv', 'nyc_taxi_clean.csv')

数据质量检核引擎 v1.0

📂 加载数据: nyc_taxi_dirty.csv
   数据形状: (2005, 7)
🔍 执行质量检核...
   VendorID非空: 31 行违反
   pickup_datetime非空: 15 行违反
   trip_distance大于0: 20 行违反
   fare_amount非负: 10 行违反
   passenger_count范围[1,6]: 12 行违反
   payment_type枚举值: 6 行违反

🔧 执行数据修复...
   删除行: 15 行
   VendorID非空: 修复 30 行
   trip_distance大于0: 修复 20 行
   fare_amount非负: 修复 10 行
   passenger_count范围[1,6]: 修复 12 行
   payment_type枚举值: 修复 6 行
   去重: 删除 5 行

💾 清洗后数据保存至: nyc_taxi_clean.csv

📄 质量报告已生成: quality_report.html

✅ 数据质量治理完成！
